
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 1 - Introduction to Spark Structured Streaming

This notebook demonstrates key concepts of Structured Streaming using practical examples with IoT sensor data.

### Objectives
- Understand stream processing fundamentals
- Work with different streaming sources and sinks
- Implement streaming transformations
- Use watermarking and windowing
- Monitor streaming queries

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## A. Stream Processing Setup

First, let's set up our streaming infrastructure and define our schema.

In [0]:
# Import necessary libraries if not already imported
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Define the schema for the stream
schema = StructType([
    StructField("customer_id", LongType(), True),
    StructField("notifications", StringType(), True),
    StructField("order_id", LongType(), True),
    StructField("order_timestamp", LongType(), True)
])

# Now use this schema for your streaming DataFrame
stream_df = spark.readStream \
    .format("json") \
    .schema(schema) \
    .option("maxFilesPerTrigger", 1) \
    .option("path", "/Volumes/dbacademy_retail/v01/retail-pipeline/orders/stream_json") \
    .load()

In [0]:
# Confirm we have set up a streaming dataframe
print(f"isStreaming: {stream_df.isStreaming}")

isStreaming: True


## B. Running Basic Streaming Queries

Now let's kick off some streaming queries.

In [0]:
# Display the stream for testing, display will start an implicit query (this is analogous to the console sink)
display(stream_df)

customer_id,notifications,order_id,order_timestamp
23094,Y,75123,1640392092
23457,N,75124,1640392500
23564,Y,75125,1640394862
23392,N,75126,1640396067
23101,Y,75127,1640399066
23466,N,75128,1640404853
23834,Y,75129,1640407272
23852,Y,75130,1640419989
23483,Y,75131,1640422131
23821,N,75132,1640423697


## C: Basic Transformations on the Stream
Stateless streaming transformations are analogous to narrow transformations we would perform on normal DataFrames (`select`, `filter`, `withColumn`, etc).

In [0]:
# Simple transformations using standard DataFrame operations
transformed_stream = stream_df \
    .withColumn("notification_status", col("notifications").isNotNull()) \
    .withColumn("order_details", concat(lit("Order #"), col("order_id").cast("string")))

# Display the transformed stream
display(transformed_stream)

customer_id,notifications,order_id,order_timestamp,notification_status,order_details
23094,Y,75123,1640392092,true,Order #75123
23457,N,75124,1640392500,true,Order #75124
23564,Y,75125,1640394862,true,Order #75125
23392,N,75126,1640396067,true,Order #75126
23101,Y,75127,1640399066,true,Order #75127
23466,N,75128,1640404853,true,Order #75128
23834,Y,75129,1640407272,true,Order #75129
23852,Y,75130,1640419989,true,Order #75130
23483,Y,75131,1640422131,true,Order #75131
23821,N,75132,1640423697,true,Order #75132


In [0]:
# Filter for only orders with notifications enabled
notifications_stream = stream_df \
    .filter(col("notifications") == "Y")

# Display filtered stream
display(notifications_stream)

customer_id,notifications,order_id,order_timestamp
23094,Y,75123,1640392092
23564,Y,75125,1640394862
23101,Y,75127,1640399066
23834,Y,75129,1640407272
23852,Y,75130,1640419989
23483,Y,75131,1640422131
23907,Y,75133,1640427687
23088,Y,75134,1640427813
23327,Y,75135,1640432499
23154,Y,75136,1640433629


In [0]:
# Stop any existing queries with the same name
for q in spark.streams.active:
    if q.name == "orders_streaming_table":
        q.stop()

# Write to memory sink for interactive querying
memory_query = stream_df.writeStream \
    .format("memory") \
    .queryName("orders_streaming_table") \
    .outputMode("append") \
    .start()

In [0]:
%sql
-- Now you can query the in-memory table using SQL
SELECT notifications, count(*) as num_notifications 
FROM orders_streaming_table GROUP BY notifications

notifications,num_notifications
Y,213
N,59


## D. Combining Multiple Streams using Union

Different stream sources can be combined using relational operators like `union`.  Let's have a look.

> NOTE: Stream-to-static DataFrame joins are fully supported and easy to implement (for example joining a stream with reference or static lookup data for enrichment). Stream-to-stream joins are supported as well, but require special handling for state management.

In [0]:
# Create a second stream with a subset of the data
filtered_stream1 = stream_df.filter(col("notifications") == "Y")
filtered_stream2 = stream_df.filter(col("notifications") == "N")

# Union the streams
combined_stream = filtered_stream1.union(filtered_stream2)

# Process the combined stream
display(combined_stream)

customer_id,notifications,order_id,order_timestamp
23094,Y,75123,1640392092
23564,Y,75125,1640394862
23101,Y,75127,1640399066
23834,Y,75129,1640407272
23852,Y,75130,1640419989
23483,Y,75131,1640422131
23907,Y,75133,1640427687
23088,Y,75134,1640427813
23327,Y,75135,1640432499
23154,Y,75136,1640433629


## E. Using Triggers to Control Processing

Triggers control how long a batch window is, let's show an example.


In [0]:
# Stop any existing queries with the same name
for q in spark.streams.active:
    if q.name == "triggered_query_table":
        q.stop()

# Process data in micro-batches every 10 seconds
triggered_query = stream_df \
    .withColumn("processing_ts", current_timestamp()) \
    .writeStream \
    .format("memory") \
    .queryName("triggered_query_table") \
    .outputMode("append") \
    .trigger(processingTime="10 seconds") \
    .start()

In [0]:
%sql
SELECT processing_ts, count(*) as count 
FROM triggered_query_table 
GROUP BY processing_ts
ORDER BY processing_ts

processing_ts,count
2025-07-01T21:47:13.688Z,174
2025-07-01T21:47:20.061Z,25
2025-07-01T21:47:30.099Z,25


## Key Takeaways

1. **Stream Processing Fundamentals**:
   - Continuous data processing
   - Schema definition
   - Basic transformations

2. **Sources and Sinks**:
   - Rate source for testing
   - Console sink for debugging
   - Memory sink for monitoring

4. **Monitoring and Management**:
   - Query monitoring
   - Progress tracking
   - Resource management


Run the cell below to stop the active streaming queries.

In [0]:
for query in spark.streams.active:
    query.stop()


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
